# Inside the challenge dataset

A hands-on tour for the Lost & Found team. Start with the dataset inventory, then inspect a real factory, both floors, equipment, connections and benchmark questions.

**Sample:** `proc-factory-dev-003`, a public CC0 procedural development building. No test answers are included. The full manifest is included, but only this building's detailed records are bundled. Source snapshot: `cb51a6f2e357db743b0c8b2b377fe902ee3ced13`.

Run all cells from top to bottom. Change `BUILDING_DIR` to explore another local building in the same format. This notebook uses Python's standard library and IPython (included with Jupyter); it needs no API key and makes no network calls.


In [1]:
from pathlib import Path
import sys, json
from collections import Counter
from IPython.display import HTML, display

# Supports launching Jupyter from the repository root or notebooks/.
NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "explore.py").exists() else Path.cwd() / "notebooks"
sys.path.insert(0, str(NOTEBOOK_DIR))
from explore import load_building, load_manifest, load_questions, table, bars, floor_svg, area, STYLE
BUILDING_DIR = NOTEBOOK_DIR / "data" / "proc-factory-dev-003"
building = load_building(BUILDING_DIR)
manifest = load_manifest()
display(HTML(f"<style>{STYLE}</style>"))
print("Loaded:", building["building"]["name"])


Loaded: Fertigung Ost 27


## 1. What data do we have?

The manifest has one row per building. Train is for development, dev is for evaluation, and test is for final generated submissions. Buildings come from Swiss Dwellings, ResPlan, OpenStreetMap indoor data, and procedural generators. Licenses differ: read `data/ATTRIBUTION.md`.


In [2]:
print(f"{len(manifest)} buildings described by the manifest")
display(HTML(bars(Counter(row["split"] for row in manifest), "Buildings by split")))
display(HTML(bars(Counter(row["source"] for row in manifest), "Buildings by source")))
display(HTML(table(manifest[:8])))


280 buildings described by the manifest


building_id,split,source,group_id,name,license,floors,levels,spaces,rooms,openings,connectors,assets,has_stairs,has_lift,has_step_free_entrance,questions,path
osm-way22744230,dev,osm_indoor,osm:way22744230,N3,ODbL-1.0,8,-1;0;1;2;3;4;5;6,325,243,386,7,890,True,True,True,9,dev/osm-way22744230
osm-way27326387,dev,osm_indoor,osm:way27326387,TUM 4107 Institutsgebäude 3,ODbL-1.0,3,-2;0;1,61,52,68,1,184,True,False,True,8,dev/osm-way27326387
osm-way27917482,dev,osm_indoor,osm:way27917482,"TUM School of Computation, Information and Technology (CIT), Departments Informatik und Mathematik",ODbL-1.0,5,-1;0;1;2;3,1623,1305,1799,32,5250,True,True,True,9,dev/osm-way27917482
osm-way360544293,dev,osm_indoor,osm:way360544293,Terminal 2,ODbL-1.0,3,0;1;2,46,22,61,8,217,False,True,True,4,dev/osm-way360544293
proc-campus-dev-000,dev,procedural,procedural:20361024,Hörsaalzentrum 024,CC0-1.0,5,0;1;2;3;4,152,116,149,6,944,True,True,True,5,dev/proc-campus-dev-000
proc-campus-dev-001,dev,procedural,procedural:20361025,Laborgebäude 025,CC0-1.0,4,0;1;2;3,97,76,95,4,582,True,True,True,5,dev/proc-campus-dev-001
proc-campus-dev-002,dev,procedural,procedural:20361026,Lehrgebäude 026,CC0-1.0,4,0;1;2;3,89,68,87,4,580,True,True,True,3,dev/proc-campus-dev-002
proc-campus-dev-006,dev,procedural,procedural:20361030,Lehrgebäude 030,CC0-1.0,4,0;1;2;3,85,72,83,2,626,True,True,True,3,dev/proc-campus-dev-006


## 2. What is inside one building?

- `building.json`: canonical metadata and spatial model.
- `equipment.csv`: flattened asset inventory, including room names and floor details.
- `floor_<level>.png/.svg` and `floor_<level>_plain.png`: visual floor plans, available upstream (not bundled).
- `questions_basic.jsonl`: warm-up lookups and routes.
- `questions.jsonl`: real benchmark tasks; train/dev include answers, test does not.

A **space** is a polygon. An **opening** joins two spaces (or a space and `outside`). A **vertical connector** links floor stops. An **asset** has a location and a `space_id`. Units are metres; X points right, Y points up. Floor 0 is ground level.


In [3]:
display(HTML(table([building["building"]])))
display(HTML(table([{"entity": key, "records": len(building[key])} for key in ("floors", "spaces", "openings", "vertical_connectors", "assets")])))
display(HTML(table(building["floors"])))
# Exact records are useful when agreeing on frontend/backend contracts.
for key in ("spaces", "openings", "vertical_connectors", "assets"):
    print("\n" + key + " — first record:")
    print(json.dumps(building[key][0], indent=2))


id,name,source,source_ref,license,attribution,split,units
proc-factory-dev-003,Fertigung Ost 27,procedural,"{'generator': 'factory', 'seed': '20361027', 'parameters': 'held_out'}",CC0-1.0,Procedurally generated for the Munich hackathon (CC0),dev,m


entity,records
floors,2
spaces,79
openings,85
vertical_connectors,2
assets,505


id,level,name,elevation_m,height_m
proc-factory-dev-003-F0,0,Ground floor,0.0,8.0
proc-factory-dev-003-F1,1,Floor 1,3.5,3.5



spaces — first record:
{
  "id": "proc-factory-dev-003-F0-S001",
  "floor_id": "proc-factory-dev-003-F0",
  "type": "hall",
  "room_type": "production_zone",
  "name": "Production zone H-A1",
  "ref": "H-A1",
  "polygon": [
    [
      0.0,
      0.0
    ],
    [
      0.0,
      17.1
    ],
    [
      -14.5,
      17.1
    ],
    [
      -14.5,
      0.0
    ]
  ],
  "synthetic": false
}

openings — first record:
{
  "id": "proc-factory-dev-003-F0-O0001",
  "floor_id": "proc-factory-dev-003-F0",
  "space_ids": [
    "proc-factory-dev-003-F0-S001",
    "proc-factory-dev-003-F0-S006"
  ],
  "kind": "opening",
  "position": [
    -14.5,
    8.55
  ],
  "width_m": 17.2,
  "step_free": true,
  "synthetic": false
}

vertical_connectors — first record:
{
  "id": "proc-factory-dev-003-V01",
  "kind": "stairs",
  "name": "Office staircase",
  "floor_ids": [
    "proc-factory-dev-003-F0",
    "proc-factory-dev-003-F1"
  ],
  "space_ids": [
    "proc-factory-dev-003-F0-S017",
    "proc-factory

## 3. Visualize the floors

Rooms are coloured by space type; blue footprints are machines/equipment; grey-green footprints are furniture/fixtures; dots are openings. Hover for exact names, IDs, door widths, accessibility and source fields. Orange dots mark openings that are not step-free. Stairs are orange regions and lifts are purple.

These are input polygons and asset footprints, not walkable paths. Display labels use approximate bounding-box centres; routing must use the official graph's representative points.


In [4]:
for floor in sorted(building["floors"], key=lambda f: f["level"]):
    print(f"{floor['name']} — level {floor['level']}, elevation {floor['elevation_m']} m")
    display(HTML(floor_svg(building, floor["id"])))


Ground floor — level 0, elevation 0.0 m


Floor 1 — level 1, elevation 3.5 m


## 4. Explore the inventory

Change `SEARCH` to find equipment by name or type. `space_id` connects each result back to a room polygon; `floor_id` connects it to a floor. Generated assets are marked separately from dataset assets.


In [5]:
display(HTML(bars(Counter(a["category"] for a in building["assets"]), "Asset categories")))
SEARCH = "cnc"
assets = [a for a in building["assets"] if SEARCH.casefold() in (a["name"] + " " + a["type"]).casefold()]
print(f"{len(assets)} matching assets")
display(HTML(table(assets, ["id", "name", "type", "space_id", "floor_id", "position", "source"])))
# Confirm the CSV is an alternative view of the same asset inventory.
import csv
with (BUILDING_DIR / "equipment.csv").open(newline="") as handle:
    equipment = list(csv.DictReader(handle))
assert {row["asset_id"] for row in equipment} == {a["id"] for a in building["assets"]}
print("CSV asset IDs match building.json:", len(equipment))


3 matching assets


id,name,type,space_id,floor_id,position,source
proc-factory-dev-003-A00001,CNC mill H-A1-A,cnc_mill,proc-factory-dev-003-F0-S001,proc-factory-dev-003-F0,"[-7.791, 8.193]",generated
proc-factory-dev-003-A00002,CNC mill H-A1-B,cnc_mill,proc-factory-dev-003-F0-S001,proc-factory-dev-003-F0,"[-2.778, 3.282]",generated
proc-factory-dev-003-A00003,CNC mill H-A1-C,cnc_mill,proc-factory-dev-003-F0-S001,proc-factory-dev-003-F0,"[-4.665, 6.385]",generated


CSV asset IDs match building.json: 505


## 5. Geometry and connectivity

Polygon areas are calculated using the shoelace formula. These are individual space areas; summing all polygons can double-count overlaps. This is a useful first tool for `largest_room`, but not enough for route, shared-wall or stacked-room questions.


In [6]:
space_rows = [{"name": s["name"], "floor_id": s["floor_id"], "type": s["type"], "room_type": s["room_type"], "area_m2": round(area(s["polygon"]), 2)} for s in building["spaces"]]
display(HTML(table(sorted(space_rows, key=lambda s: s["area_m2"], reverse=True)[:15])))
print("Connections between floors:")
display(HTML(table(building["vertical_connectors"])))
print("Entrances:")
display(HTML(table([o for o in building["openings"] if "outside" in o["space_ids"]])))
print("Door widths (first 12 doors):")
display(HTML(table([o for o in building["openings"] if o["kind"] == "door"][:12], ["id", "space_ids", "width_m", "step_free"])))


name,floor_id,type,room_type,area_m2
Production zone H-A1,proc-factory-dev-003-F0,hall,production_zone,247.95
Assembly zone H-A2,proc-factory-dev-003-F0,hall,assembly_zone,247.95
Additive zone H-A3,proc-factory-dev-003-F0,hall,additive_zone,247.95
Production zone H-A4,proc-factory-dev-003-F0,hall,production_zone,247.95
Warehouse zone H-A5,proc-factory-dev-003-F0,hall,warehouse_zone,247.95
Main aisle H-B1,proc-factory-dev-003-F0,corridor,corridor,247.95
Main aisle H-B2,proc-factory-dev-003-F0,corridor,corridor,247.95
Main aisle H-B3,proc-factory-dev-003-F0,corridor,corridor,247.95
Main aisle H-B4,proc-factory-dev-003-F0,corridor,corridor,247.95
Main aisle H-B5,proc-factory-dev-003-F0,corridor,corridor,247.95


Connections between floors:


id,kind,name,floor_ids,space_ids,step_free,synthetic
proc-factory-dev-003-V01,stairs,Office staircase,"['proc-factory-dev-003-F0', 'proc-factory-dev-003-F1']","['proc-factory-dev-003-F0-S017', 'proc-factory-dev-003-F1-S002']",False,False
proc-factory-dev-003-V02,lift,Office lift,"['proc-factory-dev-003-F0', 'proc-factory-dev-003-F1']","['proc-factory-dev-003-F0-S018', 'proc-factory-dev-003-F1-S003']",True,False


Entrances:


id,floor_id,space_ids,kind,position,width_m,step_free,synthetic
proc-factory-dev-003-F0-O0018,proc-factory-dev-003-F0,"['proc-factory-dev-003-F0-S012', 'outside']",entrance,"[-43.5, 25.65]",4.0,True,False
proc-factory-dev-003-F0-O0019,proc-factory-dev-003-F0,"['proc-factory-dev-003-F0-S013', 'outside']",entrance,"[-43.5, 42.75]",4.0,True,False
proc-factory-dev-003-F0-O0020,proc-factory-dev-003-F0,"['proc-factory-dev-003-F0-S010', 'outside']",entrance,"[-21.75, 85.5]",5.0,True,False
proc-factory-dev-003-F0-O0054,proc-factory-dev-003-F0,"['proc-factory-dev-003-F0-S016', 'outside']",entrance,"[6.0, 69.8]",1.8,True,False


Door widths (first 12 doors):


id,space_ids,width_m,step_free
proc-factory-dev-003-F0-O0021,"['proc-factory-dev-003-F0-S017', 'proc-factory-dev-003-F0-S016']",1.2,True
proc-factory-dev-003-F0-O0022,"['proc-factory-dev-003-F0-S018', 'proc-factory-dev-003-F0-S016']",1.1,True
proc-factory-dev-003-F0-O0023,"['proc-factory-dev-003-F0-S019', 'proc-factory-dev-003-F0-S016']",1.6,True
proc-factory-dev-003-F0-O0024,"['proc-factory-dev-003-F0-S019', 'proc-factory-dev-003-F0-S001']",1.6,True
proc-factory-dev-003-F0-O0025,"['proc-factory-dev-003-F0-S020', 'proc-factory-dev-003-F0-S016']",1.2,True
proc-factory-dev-003-F0-O0026,"['proc-factory-dev-003-F0-S021', 'proc-factory-dev-003-F0-S016']",1.6,True
proc-factory-dev-003-F0-O0027,"['proc-factory-dev-003-F0-S022', 'proc-factory-dev-003-F0-S016']",1.2,True
proc-factory-dev-003-F0-O0028,"['proc-factory-dev-003-F0-S023', 'proc-factory-dev-003-F0-S016']",0.9,True
proc-factory-dev-003-F0-O0029,"['proc-factory-dev-003-F0-S024', 'proc-factory-dev-003-F0-S016']",1.2,True
proc-factory-dev-003-F0-O0030,"['proc-factory-dev-003-F0-S025', 'proc-factory-dev-003-F0-S016']",0.9,True


## 6. What do we need to answer?

Warm-up questions test locating/counting/routing. The full benchmark asks for areas, width-constrained reachability, rooms above/below, shared walls, 3D distances, and the effect of adding a door. Inspect a dev question and its structured expected answer below. This notebook does not call an LLM or attempt the benchmark.


In [7]:
questions = load_questions(directory=BUILDING_DIR)
basic = load_questions("questions_basic.jsonl", directory=BUILDING_DIR)
display(HTML(table([{"set": name, "kind": kind, "count": n} for name, records in [("benchmark", questions), ("warm-up", basic)] for kind, n in Counter(q["kind"] for q in records).items()])))
display(HTML(table(questions, ["id", "kind", "question"])))
QUESTION_INDEX = 0
print(json.dumps(questions[QUESTION_INDEX], indent=2))


set,kind,count
benchmark,largest_room,1
benchmark,floor_area_total,1
benchmark,wide_delivery_count,1
benchmark,errand,1
benchmark,shared_walls_count,1
warm-up,locate,3
warm-up,count,1
warm-up,route,3
warm-up,route_step_free,1
warm-up,route_with_closure,1


id,kind,question
proc-factory-dev-003-H01,largest_room,"Which is the largest kitchen on the floor 1, and how big is it in m²? Finish with a line 'Answer: <room name or number>, <area> m²'."
proc-factory-dev-003-H02,floor_area_total,Facilities want the total floor area of every meeting room on the ground floor added together. How many square metres is that? Finish with a line 'Answer: <number>'.
proc-factory-dev-003-H03,wide_delivery_count,"I'm moving a 1.1 m wide cleaning machine. It can't go on stairs or steps (lifts are fine), and every door it passes through must be at least 1.1 m wide. It arrives through the entrance into Warehouse zone H-C3 (Ground floor). Into how many rooms on the floor 1 can it be delivered? Finish with a line 'Answer: <number>'."
proc-factory-dev-003-H04,errand,"I'm in Office 0.38 (Ground floor). I need to grab a stove from the room that has one closest to me, then carry it to the largest kitchen on the floor 1. I can carry it on the stairs. How far will I walk in total, in metres? Finish with a line 'Answer: <number>'."
proc-factory-dev-003-H05,shared_walls_count,"We're soundproofing Assembly zone H-A2 (Ground floor). How many other rooms on that floor share a wall with it? Count every room whose wall runs along it for at least 1 m, with or without a door between them; corridors, stairwells and lifts don't count. Finish with a line 'Answer: <number>'."


{
  "id": "proc-factory-dev-003-H01",
  "building_id": "proc-factory-dev-003",
  "kind": "largest_room",
  "skill": "room geometry: area",
  "question": "Which is the largest kitchen on the floor 1, and how big is it in m\u00b2? Finish with a line 'Answer: <room name or number>, <area> m\u00b2'.",
  "inputs": {
    "floor_id": "proc-factory-dev-003-F1",
    "room_type": "kitchen"
  },
  "answer": {
    "type": "space",
    "space_id": "proc-factory-dev-003-F1-S014",
    "accept": [
      "1.14",
      "Kitchen 1.14"
    ],
    "reject": [
      "1.28",
      "Kitchen 1.28",
      "1.07",
      "Kitchen 1.07",
      "1.27",
      "Kitchen 1.27"
    ],
    "area": {
      "type": "number",
      "value": 28.0,
      "unit": "m2",
      "relative_tolerance": 0.05,
      "absolute_tolerance": 0.0
    }
  }
}


## 7. Basic data integrity checks

Check unique IDs, entity references and floor membership before building on these inputs. These checks do not establish routing correctness or polygon validity.


In [8]:
space_ids = {s["id"] for s in building["spaces"]}
floor_ids = {f["id"] for f in building["floors"]}
for key in ("floors", "spaces", "openings", "vertical_connectors", "assets"):
    assert len({item["id"] for item in building[key]}) == len(building[key]), f"Duplicate IDs in {key}"
for space in building["spaces"]:
    assert space["floor_id"] in floor_ids
    assert len(space["polygon"]) >= 3 and area(space["polygon"]) > 0
for asset in building["assets"]:
    assert asset["space_id"] in space_ids and asset["floor_id"] in floor_ids
    assert next(s for s in building["spaces"] if s["id"] == asset["space_id"])["floor_id"] == asset["floor_id"]
for opening in building["openings"]:
    assert len(opening["space_ids"]) == 2
    assert opening["floor_id"] in floor_ids
    assert set(opening["space_ids"]) <= space_ids | {"outside"}
for connector in building["vertical_connectors"]:
    assert len(connector["floor_ids"]) == len(connector["space_ids"])
    for floor_id, space_id in zip(connector["floor_ids"], connector["space_ids"]):
        assert floor_id in floor_ids and space_id in space_ids
        assert next(s for s in building["spaces"] if s["id"] == space_id)["floor_id"] == floor_id
print("All sample entity IDs, references, floor memberships and basic polygon checks passed.")


All sample entity IDs, references, floor memberships and basic polygon checks passed.


## Where to go next

1. Import this real building into the Next.js viewer instead of the hand-made demo.
2. Reuse the Python starter graph: space anchors at Shapely representative points, opening nodes, parallel connections, stair cost 12 m/floor, lift cost 15 m wait + 6 m/floor. Preserve the official accessibility and closure behaviour.
3. Validate routing against dev warm-up answers; then add geometry tools for the full benchmark.
4. Connect tool results to the conversational UI with exact space/asset IDs and route coordinates.

Read [the challenge](https://github.com/Synera-GmbH/industrial-agents-hackathon-2026) and `data/ATTRIBUTION.md`. This snapshot was fetched on 2026-10-24; the HTML explorer can be regenerated with `python notebooks/explore.py`.
